# Chapter 16: RAG — The GAS Knowledge Agent

> **Part 5: Agentic AI Systems** | Guardian Angel System (GAS) — Multi-Agent Diabetic Patient Supervision

---

## 🎯 Chapter Goal

By the end of this chapter you will:

1. **Build** a 20-document medical knowledge base from ADA clinical guidelines
2. **Implement** hybrid retrieval: dense (sentence-transformers) + sparse (BM25) + RRF fusion
3. **Build** the GAS Knowledge Agent — the Advisor Agent's retrieval layer
4. **Test** on 5 clinical scenarios covering the full glucose spectrum

### Why RAG for Medical AI?

LLMs hallucinate. In a medical context, a hallucinated insulin dose recommendation could kill a patient. RAG grounds the Advisor Agent's responses in **verified clinical guidelines** — the ADA Standards of Medical Care in Diabetes.

```
Patient glucose: 280 mg/dL
        │
        ▼
  [Advisor Agent]
        │
        ├─► retrieve("hyperglycemia management") ──► ADA guideline chunk
        ├─► retrieve("correction dose calculation") ──► ISF formula
        └─► generate(context=retrieved_chunks) ──► grounded recommendation
```

## 📖 Theory Recap

### Embedding Models and Vector Similarity

**Embedding models** convert text into dense vectors in a high-dimensional space where semantically similar texts are close together.

```
"hypoglycemia treatment" ──► [0.23, -0.41, 0.87, ...] (384 dims)
"low blood sugar protocol" ──► [0.21, -0.39, 0.85, ...] (similar!)
"car engine repair" ──► [-0.71, 0.12, -0.33, ...] (very different)
```

**Cosine similarity** measures the angle between vectors:
$$\text{sim}(A, B) = \frac{A \cdot B}{\|A\| \|B\|}$$

### Chunking Strategies

| Strategy | Description | Best For |
|----------|-------------|----------|
| **Fixed-size** | Split every N tokens | Simple, predictable |
| **Semantic** | Split at sentence/paragraph boundaries | Better coherence |
| **Hierarchical** | Parent chunks + child chunks | Multi-granularity retrieval |

For medical guidelines, **semantic chunking** preserves clinical protocols (e.g., the Rule of 15 should stay in one chunk).

### Hybrid Retrieval: Dense + Sparse

**Dense retrieval** (embeddings) excels at semantic similarity but misses exact keyword matches.
**Sparse retrieval** (BM25) excels at keyword matching but misses paraphrases.

**Hybrid = best of both worlds**:
- Dense: "what should I do when glucose is low?" → finds "hypoglycemia protocol"
- Sparse: "Rule of 15" → finds exact phrase in guidelines

### Reciprocal Rank Fusion (RRF)

RRF combines ranked lists from multiple retrievers:

$$\text{RRF}(d) = \sum_{r \in \text{rankers}} \frac{1}{k + \text{rank}_r(d)}$$

where $k=60$ is a smoothing constant. Documents ranked highly by both retrievers get the highest combined score.

### Agentic RAG

In standard RAG, retrieval is fixed: always retrieve top-k for every query.

In **Agentic RAG**, the agent decides:
- **When** to retrieve (not every turn needs retrieval)
- **What** to retrieve (query reformulation, multi-hop)
- **How many** documents to retrieve (adaptive k)
- **Whether** to retrieve again (iterative refinement)

## 🔨 Build It

### Step 1: Install Dependencies

In [ ]:
# Install required packages
# %pip install sentence-transformers rank-bm25 numpy scikit-learn openai python-dotenv

### Step 2: Environment Setup

In [ ]:
import os
import json
import math
import re
import time
import numpy as np
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass, field

# TEST_MODE: set NOTEBOOK_TEST_MODE=1 to run without real API keys or heavy models
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '../..'))
from llm_utils import llm, TEST_MODE, BACKEND, MODEL

### Step 3: Build the Medical Knowledge Base

In [ ]:
# ─── Medical Knowledge Base ───────────────────────────────────────────────────
# 20 documents based on ADA Standards of Medical Care in Diabetes 2024

MEDICAL_KNOWLEDGE_BASE = [
    {
        "id": "doc_001",
        "title": "ADA Glucose Targets for Adults with Diabetes",
        "category": "targets",
        "content": (
            "The American Diabetes Association (ADA) recommends the following glucose targets for "
            "non-pregnant adults with diabetes: Fasting/preprandial glucose: 80-130 mg/dL. "
            "Postprandial glucose (1-2 hours after meal): less than 180 mg/dL. "
            "A1C target: less than 7% for most adults. Time in Range (TIR) target: greater than 70% "
            "of readings between 70-180 mg/dL. Time below range (TBR): less than 4% below 70 mg/dL, "
            "less than 1% below 54 mg/dL. Time above range (TAR): less than 25% above 180 mg/dL, "
            "less than 5% above 250 mg/dL. These targets should be individualized based on patient "
            "age, comorbidities, hypoglycemia awareness, and life expectancy."
        )
    },
    {
        "id": "doc_002",
        "title": "Hypoglycemia Treatment Protocol — Rule of 15",
        "category": "hypoglycemia",
        "content": (
            "The Rule of 15 is the standard treatment protocol for hypoglycemia (blood glucose < 70 mg/dL). "
            "Step 1: Consume 15 grams of fast-acting carbohydrates. Examples: 4 glucose tablets, "
            "4 oz (120 mL) of fruit juice or regular soda, 1 tablespoon of sugar or honey, "
            "or 3-4 pieces of hard candy. Step 2: Wait 15 minutes. Step 3: Recheck blood glucose. "
            "Step 4: If still below 70 mg/dL, repeat steps 1-3. Step 5: Once glucose is above 70 mg/dL, "
            "eat a small snack if the next meal is more than 1 hour away. "
            "For severe hypoglycemia (< 54 mg/dL) with altered consciousness: administer glucagon "
            "(1 mg IM/SC for adults) or call emergency services (911). Do NOT give oral carbohydrates "
            "to an unconscious patient."
        )
    },
    {
        "id": "doc_003",
        "title": "Hyperglycemia Management Protocol",
        "category": "hyperglycemia",
        "content": (
            "Hyperglycemia is defined as blood glucose above 180 mg/dL (postprandial) or above 130 mg/dL "
            "(fasting). Mild hyperglycemia (180-250 mg/dL): Check for missed insulin dose or meal bolus. "
            "Calculate correction dose using Insulin Sensitivity Factor (ISF): "
            "Correction dose (units) = (Current glucose - Target glucose) / ISF. "
            "Increase water intake. Recheck in 2 hours. Moderate hyperglycemia (250-350 mg/dL): "
            "Check urine or blood ketones. If ketones negative, administer correction dose. "
            "If ketones positive (> 0.6 mmol/L blood ketones), contact healthcare provider. "
            "Severe hyperglycemia (> 350 mg/dL): Seek immediate medical attention. Risk of DKA. "
            "Symptoms: excessive thirst, frequent urination, nausea, abdominal pain, fruity breath."
        )
    },
    {
        "id": "doc_004",
        "title": "Dawn Phenomenon — Morning Glucose Elevation",
        "category": "patterns",
        "content": (
            "The dawn phenomenon is a natural rise in blood glucose that occurs in the early morning hours "
            "(typically 2-8 AM) due to the release of counter-regulatory hormones (cortisol, growth hormone, "
            "glucagon) that increase insulin resistance and stimulate glucose production by the liver. "
            "It affects approximately 50% of people with T1D and 25% of people with T2D. "
            "Management strategies: For insulin pump users — increase basal rate 1-2 hours before the rise. "
            "For MDI users — consider long-acting insulin timing adjustment or adding a small correction dose "
            "at bedtime. CGM pattern analysis: Look for consistent glucose rise between 3-7 AM without "
            "preceding hypoglycemia (which would indicate Somogyi effect, not dawn phenomenon)."
        )
    },
    {
        "id": "doc_005",
        "title": "Exercise and Glucose Management",
        "category": "exercise",
        "content": (
            "Exercise affects blood glucose differently depending on type, intensity, and duration. "
            "Aerobic exercise (running, cycling, swimming): Generally lowers blood glucose during and after. "
            "Risk of hypoglycemia during exercise and up to 24 hours post-exercise. "
            "Pre-exercise glucose targets: 126-180 mg/dL for most activities. "
            "If glucose < 90 mg/dL before exercise: consume 15-30g carbs before starting. "
            "Anaerobic/high-intensity exercise (sprinting, weightlifting): May temporarily raise glucose "
            "due to catecholamine release. Post-exercise hyperglycemia is common. "
            "Insulin adjustments: Reduce bolus insulin by 25-75% for meals before prolonged exercise. "
            "For pump users: consider 50-80% temporary basal rate reduction during aerobic exercise. "
            "Post-exercise: Monitor glucose every 2-4 hours for delayed hypoglycemia risk."
        )
    },
    {
        "id": "doc_006",
        "title": "Insulin Types, Onset, Peak, and Duration",
        "category": "insulin",
        "content": (
            "Rapid-acting insulin analogs (lispro/Humalog, aspart/NovoLog, glulisine/Apidra): "
            "Onset 15 min, peak 1-2 hours, duration 3-5 hours. Use: mealtime bolus. "
            "Ultra-rapid acting (Fiasp, Lyumjev): Onset 2-5 min, peak 1-1.5 hours, duration 3-5 hours. "
            "Short-acting regular insulin (Humulin R, Novolin R): Onset 30-60 min, peak 2-4 hours, "
            "duration 5-8 hours. Intermediate-acting NPH: Onset 1-2 hours, peak 4-8 hours, "
            "duration 12-18 hours. Long-acting analogs (glargine/Lantus, detemir/Levemir): "
            "Onset 1-2 hours, no pronounced peak, duration 20-24 hours. "
            "Ultra-long acting (degludec/Tresiba): Onset 1 hour, no peak, duration > 42 hours. "
            "Insulin stacking: Avoid giving correction doses within 3-4 hours of last rapid-acting dose "
            "to prevent hypoglycemia from overlapping insulin action (insulin on board)."
        )
    },
    {
        "id": "doc_007",
        "title": "Sick Day Rules for Diabetes Management",
        "category": "sick_day",
        "content": (
            "During illness, blood glucose typically rises due to stress hormones even if eating less. "
            "Never stop insulin during illness, even if not eating. "
            "Check blood glucose every 2-4 hours. Check ketones if glucose > 250 mg/dL. "
            "Hydration: Drink 8 oz of fluid every hour. If glucose < 100 mg/dL, drink fluids with sugar. "
            "If glucose > 100 mg/dL, drink sugar-free fluids. "
            "Contact healthcare provider if: glucose > 300 mg/dL for more than 2 readings, "
            "moderate or large ketones, vomiting and unable to keep fluids down, "
            "symptoms of DKA (nausea, vomiting, abdominal pain, difficulty breathing). "
            "Medications to hold during illness: SGLT2 inhibitors (risk of euglycemic DKA), "
            "metformin (risk of lactic acidosis if dehydrated)."
        )
    },
    {
        "id": "doc_008",
        "title": "Ketone Testing Guidelines",
        "category": "ketones",
        "content": (
            "Ketone testing is essential for detecting diabetic ketoacidosis (DKA). "
            "When to test: Blood glucose > 250 mg/dL, during illness, unexplained nausea/vomiting, "
            "abdominal pain, or if using an insulin pump (missed doses). "
            "Blood ketone testing (preferred): Normal < 0.6 mmol/L. "
            "Elevated: 0.6-1.5 mmol/L — recheck in 2 hours, increase fluid intake. "
            "High: 1.5-3.0 mmol/L — contact healthcare provider, may need IV fluids. "
            "Very high: > 3.0 mmol/L — seek emergency care immediately (DKA). "
            "Urine ketone testing: Negative/trace — normal. Small (1+) — monitor. "
            "Moderate (2+) or Large (3+) — contact healthcare provider. "
            "DKA triad: hyperglycemia (> 250 mg/dL), ketosis (ketones > 1.5 mmol/L), "
            "acidosis (pH < 7.3, bicarbonate < 15 mEq/L)."
        )
    },
    {
        "id": "doc_009",
        "title": "CGM Calibration and Accuracy",
        "category": "cgm",
        "content": (
            "Continuous Glucose Monitors (CGMs) measure interstitial fluid glucose, which lags behind "
            "blood glucose by 5-15 minutes. This lag is most significant during rapid glucose changes. "
            "Factory-calibrated CGMs (Dexcom G7, Libre 3): Do not require fingerstick calibration. "
            "MARD (Mean Absolute Relative Difference): Dexcom G7 ~8.2%, Libre 3 ~7.8%. "
            "When to verify with fingerstick: Symptoms don't match CGM reading, "
            "glucose is rapidly changing (> 2 mg/dL/min), before making treatment decisions "
            "during hypoglycemia, CGM shows compression artifact (lying on sensor). "
            "CGM alarms: Set low alarm at 70 mg/dL (or 80 mg/dL for hypoglycemia unawareness). "
            "Set urgent low alarm at 55 mg/dL. Set high alarm at 250 mg/dL. "
            "Predictive alarms: Alert 20-30 minutes before projected low/high."
        )
    },
    {
        "id": "doc_010",
        "title": "Carbohydrate Counting Basics",
        "category": "nutrition",
        "content": (
            "Carbohydrate counting is the primary meal planning method for insulin-dependent diabetes. "
            "Insulin-to-carb ratio (ICR): Units of insulin needed per gram of carbohydrate. "
            "Example: ICR of 1:15 means 1 unit of insulin covers 15g of carbs. "
            "Mealtime bolus = Total carbs (g) / ICR. "
            "Foods that raise glucose: All carbohydrates (grains, fruits, dairy, legumes, starchy vegetables). "
            "Fiber: Subtract fiber from total carbs for net carbs (fiber doesn't raise glucose). "
            "Sugar alcohols: Count half the grams (partial glucose impact). "
            "Glycemic index (GI): High-GI foods (white bread, glucose tablets) raise glucose faster. "
            "Low-GI foods (legumes, most vegetables) raise glucose more slowly. "
            "Fat and protein: High-fat meals delay glucose absorption (pizza effect — late glucose rise). "
            "High-protein meals can raise glucose 3-5 hours later via gluconeogenesis."
        )
    },
    {
        "id": "doc_011",
        "title": "Hypoglycemia Unawareness — Recognition and Management",
        "category": "hypoglycemia",
        "content": (
            "Hypoglycemia unawareness (HU) is the inability to recognize hypoglycemia symptoms, "
            "occurring in 20-25% of people with T1D and increasing with diabetes duration. "
            "Cause: Repeated hypoglycemia blunts counter-regulatory hormone responses and "
            "autonomic warning symptoms. Risk factors: Long diabetes duration, tight glycemic control, "
            "history of severe hypoglycemia, autonomic neuropathy. "
            "Management: Blood Glucose Awareness Training (BGAT). "
            "Strict avoidance of hypoglycemia for 2-3 weeks can partially restore awareness. "
            "Raise glucose targets temporarily (A1C 7.5-8%). "
            "CGM with low alarms is essential — set alarm at 80 mg/dL. "
            "Continuous glucose monitoring reduces severe hypoglycemia by 72% in HU patients. "
            "Glucagon kit: Ensure caregiver knows how to administer. "
            "Nasal glucagon (Baqsimi) or auto-injector (Gvoke) preferred for ease of use."
        )
    },
    {
        "id": "doc_012",
        "title": "Insulin Pump Therapy — Basal Rate Management",
        "category": "insulin_pump",
        "content": (
            "Insulin pump therapy (CSII) delivers rapid-acting insulin continuously via a subcutaneous "
            "catheter. Basal rate: Background insulin delivery (units/hour) that maintains glucose "
            "stability between meals and overnight. Total daily dose (TDD): Approximately 50% basal, "
            "50% bolus for most pump users. Basal rate testing: Fast for 4-6 hours, check glucose "
            "every 1-2 hours. If glucose rises > 30 mg/dL, increase basal rate by 10-20%. "
            "If glucose falls > 30 mg/dL, decrease basal rate by 10-20%. "
            "Temporary basal rates: Reduce by 50-80% for exercise. "
            "Increase by 20-50% for illness or high-carb meals. "
            "Infusion site: Rotate sites to prevent lipohypertrophy. Change every 2-3 days. "
            "Unexplained hyperglycemia on pump: Check infusion site first (kinked cannula, "
            "site failure). If glucose > 300 mg/dL, give correction via injection (not pump) "
            "and change infusion set."
        )
    },
    {
        "id": "doc_013",
        "title": "Nocturnal Hypoglycemia — Prevention and Detection",
        "category": "hypoglycemia",
        "content": (
            "Nocturnal hypoglycemia (NH) occurs during sleep, often undetected, and is responsible for "
            "up to 50% of severe hypoglycemia episodes in T1D. Risk factors: Glucose < 100 mg/dL at "
            "bedtime, recent exercise, alcohol consumption, missed bedtime snack. "
            "Prevention: Target bedtime glucose of 100-140 mg/dL. "
            "Bedtime snack with protein and complex carbs if glucose < 120 mg/dL. "
            "Reduce evening basal rate if pattern of nocturnal lows. "
            "CGM with overnight alarms: Essential for patients with HU. "
            "Closed-loop systems (artificial pancreas): Automatically suspend insulin during predicted lows. "
            "Symptoms of NH: Nightmares, night sweats, morning headache, unexplained morning hyperglycemia "
            "(Somogyi effect — rebound hyperglycemia after nocturnal low). "
            "If NH suspected: Check glucose at 2-3 AM for 3 consecutive nights."
        )
    },
    {
        "id": "doc_014",
        "title": "Diabetic Ketoacidosis (DKA) — Recognition and Emergency Response",
        "category": "emergency",
        "content": (
            "DKA is a life-threatening complication of diabetes caused by insulin deficiency. "
            "Diagnostic criteria: Blood glucose > 250 mg/dL (may be lower in euglycemic DKA), "
            "blood ketones > 3.0 mmol/L or urine ketones moderate/large, "
            "arterial pH < 7.3 or bicarbonate < 15 mEq/L. "
            "Symptoms: Polyuria, polydipsia, nausea, vomiting, abdominal pain, "
            "Kussmaul breathing (deep, rapid), fruity breath odor, altered consciousness. "
            "Emergency response: Call 911 immediately. Do NOT attempt home treatment of DKA. "
            "Hospital treatment: IV fluids (0.9% NaCl), IV insulin infusion, potassium replacement. "
            "Triggers: Infection (most common), missed insulin doses, new-onset T1D, "
            "SGLT2 inhibitor use, pump failure. "
            "Prevention: Never skip insulin, check ketones when glucose > 250 mg/dL, "
            "sick day rules, pump site checks."
        )
    },
    {
        "id": "doc_015",
        "title": "Insulin Sensitivity Factor (ISF) Calculation",
        "category": "insulin",
        "content": (
            "The Insulin Sensitivity Factor (ISF), also called correction factor, estimates how much "
            "1 unit of rapid-acting insulin will lower blood glucose. "
            "Rule of 1800 (for rapid-acting insulin): ISF = 1800 / Total Daily Dose (TDD). "
            "Example: TDD = 36 units/day → ISF = 1800/36 = 50 mg/dL per unit. "
            "Rule of 1500 (for regular insulin): ISF = 1500 / TDD. "
            "Correction dose formula: Correction dose = (Current BG - Target BG) / ISF. "
            "Example: Current BG = 280, Target BG = 120, ISF = 50 → Correction = (280-120)/50 = 3.2 units. "
            "Safety rules: Never give correction within 3-4 hours of last rapid-acting dose "
            "(insulin on board). Round down for safety. Maximum single correction dose: 10% of TDD. "
            "ISF varies by time of day: typically lower in morning (more insulin resistant), "
            "higher in afternoon."
        )
    },
    {
        "id": "doc_016",
        "title": "Alcohol and Blood Glucose Management",
        "category": "lifestyle",
        "content": (
            "Alcohol has complex effects on blood glucose in diabetes. "
            "Mechanism: Alcohol inhibits hepatic gluconeogenesis (liver's ability to release glucose), "
            "increasing hypoglycemia risk, especially when fasting. "
            "Initial effect: Beer and sweet cocktails may raise glucose (carbohydrate content). "
            "Delayed effect: Hypoglycemia risk 6-12 hours after drinking, including overnight. "
            "Safe drinking guidelines: Never drink on an empty stomach. "
            "Eat carbohydrates with alcohol. Limit to 1-2 drinks. "
            "Check glucose before bed — target 140-180 mg/dL if drinking. "
            "Set CGM alarm at 100 mg/dL overnight. "
            "Do NOT reduce basal insulin when drinking (hypoglycemia risk). "
            "Wear medical ID. Inform companions about hypoglycemia symptoms "
            "(may be mistaken for intoxication)."
        )
    },
    {
        "id": "doc_017",
        "title": "Pregnancy and Diabetes — Glucose Targets",
        "category": "special_populations",
        "content": (
            "Glucose management during pregnancy requires tighter targets to prevent fetal complications. "
            "ADA targets for pregnant women with preexisting diabetes: "
            "Fasting glucose: 70-95 mg/dL. 1-hour postprandial: < 140 mg/dL. "
            "2-hour postprandial: < 120 mg/dL. A1C: < 6% if achievable without hypoglycemia. "
            "CGM use in pregnancy: Strongly recommended for T1D. "
            "TIR target: > 70% in range 63-140 mg/dL. TBR < 4% below 63 mg/dL. "
            "Insulin requirements change dramatically: Decrease in first trimester (nausea, vomiting), "
            "increase significantly in second and third trimesters (placental hormones increase resistance). "
            "Hypoglycemia risk: Highest in first trimester. Ensure glucagon kit available. "
            "Postpartum: Insulin requirements drop dramatically after delivery — reduce doses by 50%."
        )
    },
    {
        "id": "doc_018",
        "title": "Continuous Glucose Monitor Trends and Arrows",
        "category": "cgm",
        "content": (
            "CGM trend arrows indicate the rate and direction of glucose change. "
            "Trend arrow interpretation: "
            "↑↑ (rapidly rising): > 3 mg/dL/min — glucose rising very fast. "
            "↑ (rising): 2-3 mg/dL/min — glucose rising quickly. "
            "↗ (slowly rising): 1-2 mg/dL/min — glucose rising moderately. "
            "→ (stable): < 1 mg/dL/min — glucose stable. "
            "↘ (slowly falling): 1-2 mg/dL/min — glucose falling moderately. "
            "↓ (falling): 2-3 mg/dL/min — glucose falling quickly. "
            "↓↓ (rapidly falling): > 3 mg/dL/min — glucose falling very fast. "
            "Treatment adjustments based on trend: Add 15g carbs if glucose 80-100 mg/dL with ↓ arrow. "
            "Add 30g carbs if glucose 80-100 mg/dL with ↓↓ arrow. "
            "Reduce meal bolus by 20% if glucose rising rapidly before meal. "
            "Trend-adjusted correction: Multiply correction dose by 0.75 if glucose falling, "
            "1.25 if glucose rising rapidly."
        )
    },
    {
        "id": "doc_019",
        "title": "Glucagon Emergency Kit — Administration Guide",
        "category": "emergency",
        "content": (
            "Glucagon is the emergency treatment for severe hypoglycemia when the patient cannot "
            "safely consume oral carbohydrates. Indications: Unconscious, seizure, unable to swallow, "
            "glucose < 54 mg/dL with altered mental status. "
            "Traditional glucagon kit (IM/SC injection): Mix powder with diluent. "
            "Adult dose: 1 mg (full kit). Child < 44 lbs: 0.5 mg. "
            "Inject into outer thigh or upper arm. Patient should respond in 10-15 minutes. "
            "Nasal glucagon (Baqsimi): 3 mg powder in one nostril. No mixing required. "
            "Auto-injector (Gvoke): Pre-filled, inject into outer thigh. "
            "After glucagon: Patient may vomit — position on side. "
            "Once conscious, give 15-20g fast carbs. Call 911 if no response in 15 minutes. "
            "Caregiver training: All caregivers of T1D patients should be trained in glucagon use. "
            "Store at room temperature, check expiration date annually."
        )
    },
    {
        "id": "doc_020",
        "title": "Closed-Loop Insulin Delivery — Artificial Pancreas Systems",
        "category": "technology",
        "content": (
            "Closed-loop insulin delivery (artificial pancreas) automatically adjusts insulin delivery "
            "based on CGM readings using a control algorithm. "
            "FDA-approved systems: Tandem Control-IQ, Omnipod 5, Medtronic MiniMed 780G. "
            "How it works: CGM reading → algorithm → pump adjusts basal rate every 5 minutes. "
            "Benefits: Reduces hypoglycemia by 60-70%, improves TIR by 10-15 percentage points, "
            "reduces A1C by 0.3-0.5%, improves sleep quality. "
            "Limitations: Still requires meal announcements (bolus for meals). "
            "Algorithm cannot predict exercise-induced lows without user input. "
            "Sensor accuracy affects algorithm performance. "
            "User still needs to understand diabetes management for system failures. "
            "Hybrid closed-loop: User announces meals, system handles basal. "
            "Full closed-loop (investigational): No meal announcements required."
        )
    }
]

print(f"✅ Medical knowledge base loaded: {len(MEDICAL_KNOWLEDGE_BASE)} documents")
print("\nDocument categories:")
from collections import Counter
cats = Counter(d["category"] for d in MEDICAL_KNOWLEDGE_BASE)
for cat, count in sorted(cats.items()):
    print(f"  {cat}: {count} doc(s)")

### Step 4: Build the Dense Embedding Index

In [ ]:
# ─── Dense Embedding with sentence-transformers ───────────────────────────────

class DenseRetriever:
    """Dense retrieval using sentence-transformers embeddings."""

    def __init__(self, documents: List[dict], model_name: str = "all-MiniLM-L6-v2", test_mode: bool = False):
        self.documents = documents
        self.test_mode = test_mode
        self.model = None
        self.embeddings = None

        if test_mode:
            print("🧪 TEST_MODE: Using random mock embeddings (384 dims)")
            np.random.seed(42)
            self.embeddings = np.random.randn(len(documents), 384).astype(np.float32)
            # Normalize
            norms = np.linalg.norm(self.embeddings, axis=1, keepdims=True)
            self.embeddings = self.embeddings / norms
        else:
            try:
                from sentence_transformers import SentenceTransformer
                print(f"Loading model: {model_name}...")
                self.model = SentenceTransformer(model_name)
                texts = [f"{d['title']}. {d['content']}" for d in documents]
                self.embeddings = self.model.encode(texts, normalize_embeddings=True, show_progress_bar=True)
                print(f"✅ Embedded {len(documents)} documents ({self.embeddings.shape[1]} dims)")
            except ImportError:
                print("⚠️  sentence-transformers not installed — falling back to mock embeddings")
                np.random.seed(42)
                self.embeddings = np.random.randn(len(documents), 384).astype(np.float32)
                norms = np.linalg.norm(self.embeddings, axis=1, keepdims=True)
                self.embeddings = self.embeddings / norms
                self.test_mode = True

    def embed_query(self, query: str) -> np.ndarray:
        """Embed a query string."""
        if self.test_mode or self.model is None:
            # Deterministic mock: hash query to seed
            seed = sum(ord(c) for c in query) % 10000
            np.random.seed(seed)
            vec = np.random.randn(384).astype(np.float32)
            return vec / np.linalg.norm(vec)
        return self.model.encode([query], normalize_embeddings=True)[0]

    def retrieve(self, query: str, k: int = 5) -> List[Tuple[int, float]]:
        """Return top-k (doc_index, score) pairs."""
        q_vec = self.embed_query(query)
        scores = self.embeddings @ q_vec  # cosine similarity (normalized)
        top_k_idx = np.argsort(scores)[::-1][:k]
        return [(int(idx), float(scores[idx])) for idx in top_k_idx]


# Build dense index
dense_retriever = DenseRetriever(MEDICAL_KNOWLEDGE_BASE, test_mode=TEST_MODE)

# Quick test
test_results = dense_retriever.retrieve("hypoglycemia treatment", k=3)
print("\nDense retrieval test — 'hypoglycemia treatment':")
for idx, score in test_results:
    doc = MEDICAL_KNOWLEDGE_BASE[idx]
    print(f"  [{score:.3f}] {doc['title']}")

### Step 5: Build the BM25 Sparse Index

In [ ]:
# ─── BM25 Sparse Retrieval ────────────────────────────────────────────────────

class BM25Retriever:
    """BM25 sparse retrieval."""

    def __init__(self, documents: List[dict]):
        self.documents = documents
        self.bm25 = None
        self._build_index()

    def _tokenize(self, text: str) -> List[str]:
        """Simple whitespace + lowercase tokenizer."""
        text = text.lower()
        text = re.sub(r'[^a-z0-9\s]', ' ', text)
        return text.split()

    def _build_index(self):
        try:
            from rank_bm25 import BM25Okapi
            corpus = [self._tokenize(f"{d['title']} {d['content']}") for d in self.documents]
            self.bm25 = BM25Okapi(corpus)
            print(f"✅ BM25 index built: {len(self.documents)} documents")
        except ImportError:
            print("⚠️  rank-bm25 not installed — using mock BM25")
            self.bm25 = None

    def retrieve(self, query: str, k: int = 5) -> List[Tuple[int, float]]:
        """Return top-k (doc_index, score) pairs."""
        if self.bm25 is None:
            # Mock: keyword matching
            query_words = set(query.lower().split())
            scores = []
            for i, doc in enumerate(self.documents):
                doc_words = set((doc['title'] + ' ' + doc['content']).lower().split())
                overlap = len(query_words & doc_words)
                scores.append((i, float(overlap)))
            scores.sort(key=lambda x: x[1], reverse=True)
            return scores[:k]

        tokens = self._tokenize(query)
        scores = self.bm25.get_scores(tokens)
        top_k_idx = np.argsort(scores)[::-1][:k]
        return [(int(idx), float(scores[idx])) for idx in top_k_idx]


# Build BM25 index
bm25_retriever = BM25Retriever(MEDICAL_KNOWLEDGE_BASE)

# Quick test
test_results_bm25 = bm25_retriever.retrieve("Rule of 15 hypoglycemia", k=3)
print("\nBM25 retrieval test — 'Rule of 15 hypoglycemia':")
for idx, score in test_results_bm25:
    doc = MEDICAL_KNOWLEDGE_BASE[idx]
    print(f"  [{score:.3f}] {doc['title']}")

### Step 6: Implement Reciprocal Rank Fusion (RRF)

In [ ]:
# ─── Reciprocal Rank Fusion ───────────────────────────────────────────────────

def reciprocal_rank_fusion(
    ranked_lists: List[List[Tuple[int, float]]],
    k: int = 60
) -> List[Tuple[int, float]]:
    """
    Combine multiple ranked lists using Reciprocal Rank Fusion.

    RRF score = sum over rankers of: 1 / (k + rank)
    where k=60 is a smoothing constant (Cormack et al., 2009).

    Args:
        ranked_lists: List of ranked result lists, each as [(doc_idx, score), ...]
        k: Smoothing constant (default 60)

    Returns:
        Fused ranked list as [(doc_idx, rrf_score), ...]
    """
    rrf_scores: Dict[int, float] = {}

    for ranked_list in ranked_lists:
        for rank, (doc_idx, _) in enumerate(ranked_list, start=1):
            rrf_scores[doc_idx] = rrf_scores.get(doc_idx, 0.0) + 1.0 / (k + rank)

    # Sort by RRF score descending
    fused = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    return fused


class HybridRetriever:
    """Hybrid retrieval: dense + BM25 fused with RRF."""

    def __init__(self, documents: List[dict], dense: DenseRetriever, sparse: BM25Retriever):
        self.documents = documents
        self.dense = dense
        self.sparse = sparse

    def retrieve(self, query: str, k: int = 5, fetch_k: int = 10) -> List[dict]:
        """
        Retrieve top-k documents using hybrid dense+sparse+RRF.

        Args:
            query: Search query
            k: Number of final results
            fetch_k: Number of candidates from each retriever

        Returns:
            List of document dicts with added 'rrf_score' field
        """
        # Get candidates from each retriever
        dense_results = self.dense.retrieve(query, k=fetch_k)
        sparse_results = self.sparse.retrieve(query, k=fetch_k)

        # Fuse with RRF
        fused = reciprocal_rank_fusion([dense_results, sparse_results])

        # Return top-k documents
        results = []
        for doc_idx, rrf_score in fused[:k]:
            doc = dict(self.documents[doc_idx])
            doc["rrf_score"] = round(rrf_score, 6)
            results.append(doc)

        return results


# Build hybrid retriever
hybrid_retriever = HybridRetriever(MEDICAL_KNOWLEDGE_BASE, dense_retriever, bm25_retriever)

# Test hybrid retrieval
print("Hybrid retrieval test — 'Rule of 15 hypoglycemia treatment':")
results = hybrid_retriever.retrieve("Rule of 15 hypoglycemia treatment", k=3)
for r in results:
    print(f"  [RRF={r['rrf_score']:.5f}] {r['title']}")
    print(f"    {r['content'][:100]}...")

### Step 7: Build the GAS Knowledge Agent

In [ ]:
# ─── GAS Knowledge Agent ─────────────────────────────────────────────────────

class GASKnowledgeAgent:
    """
    The GAS Advisor Agent's knowledge layer.
    Takes a clinical question, retrieves relevant guidelines, generates a grounded answer.
    """

    def __init__(self, retriever: HybridRetriever, test_mode: bool = False):
        self.retriever = retriever
        self.test_mode = test_mode

    def _build_context(self, docs: List[dict]) -> str:
        """Format retrieved documents as context for the LLM."""
        context_parts = []
        for i, doc in enumerate(docs, 1):
            context_parts.append(
                f"[Source {i}: {doc['title']}]\n{doc['content']}"
            )
        return "\n\n".join(context_parts)

    def _generate_mock_answer(self, question: str, context: str, glucose_value: float = None) -> str:
        """Generate a mock answer based on retrieved context (no LLM needed)."""
        # Extract key information from context
        if glucose_value and glucose_value < 54:
            return (
                f"Based on ADA guidelines, glucose of {glucose_value} mg/dL represents SEVERE HYPOGLYCEMIA. "
                "IMMEDIATE ACTION REQUIRED: If conscious, administer 15-20g fast-acting carbohydrates "
                "(4 glucose tablets, 4oz juice). If unconscious, administer glucagon and call 911. "
                "Recheck glucose in 15 minutes (Rule of 15). Do NOT give insulin."
            )
        elif glucose_value and glucose_value < 70:
            return (
                f"Glucose of {glucose_value} mg/dL is below target (70 mg/dL). "
                "Apply Rule of 15: consume 15g fast-acting carbs, wait 15 minutes, recheck. "
                "If still below 70, repeat. Once above 70, eat a small snack if next meal > 1 hour away."
            )
        elif glucose_value and glucose_value > 250:
            return (
                f"Glucose of {glucose_value} mg/dL indicates significant hyperglycemia. "
                "Check ketones immediately. If ketones < 0.6 mmol/L, calculate correction dose: "
                "(Current BG - Target BG) / ISF. Increase water intake. Recheck in 2 hours. "
                "If ketones > 1.5 mmol/L, contact healthcare provider."
            )
        else:
            # Generic: extract first relevant sentence from context
            first_doc = context.split('\n')[1] if '\n' in context else context[:200]
            return f"Based on clinical guidelines: {first_doc[:300]}..."

    def _generate_live_answer(self, question: str, context: str) -> str:
        """Generate answer using LLM (Ollama/OpenAI via llm_utils)."""
        system_prompt = (
            "You are the GAS Advisor Agent — a clinical decision support AI for diabetic patient monitoring. "
            "Answer questions using ONLY the provided clinical guidelines. "
            "Be precise, cite sources, and always prioritize patient safety. "
            "If the situation is an emergency, say so clearly."
        )
        user_prompt = f"""Clinical question: {question}

Retrieved guidelines:
{context}

Provide a concise, actionable clinical recommendation based on these guidelines."""
        return llm.chat(
            [{"role": "user", "content": user_prompt}],
            system=system_prompt,
            temperature=0.1,
            max_tokens=300,
        )

    def answer(self, question: str, k: int = 3, glucose_value: float = None) -> dict:
        """
        Answer a clinical question using RAG.

        Returns:
            dict with 'question', 'retrieved_docs', 'answer', 'sources'
        """
        # Retrieve relevant documents
        docs = self.retriever.retrieve(question, k=k)
        context = self._build_context(docs)

        # Generate answer
        if self.test_mode:
            answer = self._generate_mock_answer(question, context, glucose_value)
        else:
            try:
                answer = self._generate_live_answer(question, context)
            except Exception as e:
                answer = self._generate_mock_answer(question, context, glucose_value)
                answer += f" [Note: Live LLM unavailable: {e}]"

        return {
            "question": question,
            "retrieved_docs": [{"title": d["title"], "rrf_score": d["rrf_score"]} for d in docs],
            "answer": answer,
            "sources": [d["title"] for d in docs]
        }


# Initialize the GAS Knowledge Agent
knowledge_agent = GASKnowledgeAgent(hybrid_retriever, test_mode=TEST_MODE)
print("✅ GAS Knowledge Agent initialized")

### Step 8: Test on 5 Clinical Scenarios

In [ ]:
# ─── 5 Clinical Test Scenarios ────────────────────────────────────────────────

CLINICAL_SCENARIOS = [
    {
        "scenario": "Critical Hypoglycemia",
        "question": "Patient glucose is 45 mg/dL with a falling trend. What is the immediate treatment?",
        "glucose": 45.0
    },
    {
        "scenario": "Mild Hypoglycemia",
        "question": "Glucose is 62 mg/dL. Patient is conscious and can swallow. Apply Rule of 15.",
        "glucose": 62.0
    },
    {
        "scenario": "Significant Hyperglycemia",
        "question": "Glucose is 280 mg/dL, 2 hours after dinner. No ketones. Calculate correction dose for ISF=50, target=120.",
        "glucose": 280.0
    },
    {
        "scenario": "Pre-Exercise Check",
        "question": "Patient wants to go for a 45-minute run. Glucose is 88 mg/dL. What should they do?",
        "glucose": 88.0
    },
    {
        "scenario": "Nocturnal Pattern",
        "question": "CGM shows glucose dropping from 140 to 95 mg/dL between 2-4 AM for 3 consecutive nights. What is the likely cause and management?",
        "glucose": 95.0
    }
]

print("🏥 GAS Knowledge Agent — Clinical Scenario Testing")
print("=" * 65)

all_results = []
for i, scenario in enumerate(CLINICAL_SCENARIOS, 1):
    print(f"\n{'─'*65}")
    print(f"Scenario {i}: {scenario['scenario']}")
    print(f"Question: {scenario['question'][:80]}...")

    result = knowledge_agent.answer(
        scenario["question"],
        k=3,
        glucose_value=scenario["glucose"]
    )

    print(f"\nRetrieved docs:")
    for doc in result["retrieved_docs"]:
        print(f"  [RRF={doc['rrf_score']:.5f}] {doc['title']}")

    print(f"\nAnswer: {result['answer'][:200]}...")
    all_results.append(result)

print(f"\n{'='*65}")
print(f"✅ Tested {len(CLINICAL_SCENARIOS)} clinical scenarios")

### Step 9: Agentic RAG — The Agent Decides When to Retrieve

In [ ]:
# ─── Agentic RAG: Adaptive Retrieval ─────────────────────────────────────────

def agentic_rag_decision(glucose_value: float, trend: str, context_available: bool) -> dict:
    """
    Decide whether to retrieve, what to retrieve, and how many documents.
    This is the 'agentic' part of Agentic RAG.
    """
    decision = {
        "should_retrieve": True,
        "query": "",
        "k": 3,
        "reason": ""
    }

    # Critical situations: retrieve more, with specific queries
    if glucose_value < 54:
        decision["query"] = "severe hypoglycemia emergency treatment glucagon Rule of 15"
        decision["k"] = 5  # More context for emergencies
        decision["reason"] = "Critical hypoglycemia — retrieve emergency protocols"

    elif glucose_value < 70:
        decision["query"] = "hypoglycemia treatment Rule of 15 fast-acting carbohydrates"
        decision["k"] = 3
        decision["reason"] = "Hypoglycemia — retrieve treatment protocol"

    elif 70 <= glucose_value <= 180:
        if context_available:
            # Normal range with context — no retrieval needed
            decision["should_retrieve"] = False
            decision["reason"] = "Normal glucose range — no retrieval needed"
        else:
            decision["query"] = "ADA glucose targets normal range management"
            decision["k"] = 2
            decision["reason"] = "Normal range — retrieve targets for context"

    elif glucose_value <= 250:
        if trend == "rising":
            decision["query"] = "hyperglycemia correction dose insulin sensitivity factor"
        else:
            decision["query"] = "hyperglycemia management monitoring"
        decision["k"] = 3
        decision["reason"] = f"Hyperglycemia ({trend}) — retrieve correction protocol"

    else:  # > 250
        decision["query"] = "severe hyperglycemia DKA ketones emergency"
        decision["k"] = 5
        decision["reason"] = "Severe hyperglycemia — retrieve DKA protocol"

    return decision


# Demo: Agentic RAG decisions for different glucose levels
test_cases = [
    (45, "falling", False),
    (65, "stable", False),
    (120, "stable", True),   # Normal + context available → skip retrieval
    (220, "rising", False),
    (310, "stable", False),
]

print("🤖 Agentic RAG — Retrieval Decisions")
print("=" * 60)
for glucose, trend, has_context in test_cases:
    decision = agentic_rag_decision(glucose, trend, has_context)
    retrieve_str = f"k={decision['k']}" if decision["should_retrieve"] else "SKIP"
    print(f"\nGlucose={glucose} mg/dL, trend={trend}, context={has_context}")
    print(f"  → Retrieve: {retrieve_str}")
    print(f"  → Reason: {decision['reason']}")
    if decision["should_retrieve"]:
        print(f"  → Query: '{decision['query'][:60]}...'")

## 🏥 GAS Connection

What we built in this chapter **IS the GAS Advisor Agent's knowledge layer**.

Here's how it fits into the full GAS pipeline:

```
Monitor Agent detects: glucose = 280 mg/dL
        │
        ▼
Supervisor routes to: Advisor Agent
        │
        ▼
Advisor Agent:
  1. agentic_rag_decision(280, "rising") → retrieve k=3
  2. hybrid_retriever.retrieve("hyperglycemia correction dose ISF", k=3)
     → [Hyperglycemia Management, ISF Calculation, Insulin Types]
  3. knowledge_agent.answer(question, context=retrieved_docs)
     → "Correction dose = (280-120)/50 = 3.2 units. Check ketones first."
        │
        ▼
Safety Critic validates the recommendation (Chapter 18)
```

### Why Hybrid Retrieval Matters for Medical AI

| Query Type | Dense Wins | BM25 Wins |
|------------|-----------|----------|
| "what to do when glucose is low" | ✅ Semantic match | ❌ No keyword match |
| "Rule of 15" | ❌ Misses exact phrase | ✅ Exact keyword match |
| "ISF calculation formula" | ✅ Finds related content | ✅ Finds exact term |
| "Kussmaul breathing" | ❌ Rare term | ✅ Exact match |

RRF ensures we get the best of both worlds — critical for medical applications where both semantic understanding AND exact protocol names matter.

## ✅ Checkpoint

### What We Built

| Component | Description |
|-----------|-------------|
| `MEDICAL_KNOWLEDGE_BASE` | 20 ADA-based clinical guideline documents |
| `DenseRetriever` | Sentence-transformer embeddings (all-MiniLM-L6-v2) |
| `BM25Retriever` | Sparse keyword retrieval |
| `reciprocal_rank_fusion()` | RRF fusion of multiple ranked lists |
| `HybridRetriever` | Dense + BM25 + RRF combined |
| `GASKnowledgeAgent` | Full RAG agent with clinical Q&A |
| `agentic_rag_decision()` | Adaptive retrieval — agent decides when/what to retrieve |

### Key Concepts

- **Dense retrieval**: Semantic similarity via embeddings
- **BM25**: Keyword-based sparse retrieval
- **RRF**: Principled fusion of multiple ranked lists
- **Agentic RAG**: Agent controls retrieval strategy based on context

### What's Next

**Chapter 17: Memory** — We'll give the Advisor Agent long-term memory:
- Working memory: last 10 glucose readings in context
- Episodic memory: past hypoglycemia episodes in vector store
- Semantic memory: patient profile knowledge graph
- Memory consolidation and temporal decay

---
*Guardian Angel System | Part 5: Agentic AI | Chapter 16*